# Notebook 6 — LightGBM LambdaRank

#### Goal
- FAISS retrieved candidates.
- Notebook 5 created features.
- Now LightGBM learns:
  "Which candidate should appear first?"

### Pipeline

### Install LighGBM

In [2]:
!pip install lightgbm

### import required libraries

In [3]:
import pandas as pd
import numpy as np
import pickle
import faiss

from sklearn.metrics.pairwise import cosine_similarity
from lightgbm import LGBMRanker

### Load Feature Dataset

In [4]:
items_df = pd.read_csv("data/item_df.csv")
interactions_df = pd.read_csv("data/interactions.csv")

with open("data/item_embeddings.pkl", "rb") as f:
    item_embeddings = pickle.load(f)

with open("data/user_embeddings.pkl", "rb") as f:
    user_embeddings = pickle.load(f)

### Create Item Dictionary

In [5]:
item_embedding_dict = {}

for idx, row in items_df.iterrows():
    item_embedding_dict[row["item_id"]] = item_embeddings[idx]

### Build FAISS Index

In [6]:
item_embeddings = np.array(item_embeddings).astype("float32")

dimension = item_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(item_embeddings)

### Popularity Feature

In [7]:
popularity = interactions_df.groupby(
    "item_id"
).size()

### Create Training Rows

In [8]:
training_rows = []
groups = []

### Loop Through Every User

In [9]:
all_users = list(user_embeddings.keys())

In [10]:
for user_id in all_users:

    user_vector = np.array(
        user_embeddings[user_id]
    ).astype("float32")

    user_vector = user_vector.reshape(1, -1)

    D, I = index.search(
        user_vector,
        k=20
    )

    groups.append(20)

    for idx in I[0]:

        item_row = items_df.iloc[idx]

        item_id = item_row["item_id"]

        item_vector = item_embedding_dict[item_id]

        similarity_score = cosine_similarity(
            user_vector,
            item_vector.reshape(1,-1)
        )[0][0]

        popularity_score = popularity.get(
            item_id,
            0
        )

        recency_score = np.random.randint(1,30)

        preference_score = np.random.uniform(
            0.5,
            1.0
        )

        label = np.random.choice(
            [0,1],
            p=[0.7,0.3]
        )

        training_rows.append([
            user_id,
            item_id,
            similarity_score,
            popularity_score,
            recency_score,
            preference_score,
            label
        ])

### Create DataFrame

In [11]:
ranking_df = pd.DataFrame(
    training_rows,
    columns=[
        "user_id",
        "item_id",
        "similarity_score",
        "popularity_score",
        "recency_score",
        "preference_score",
        "label"
    ]
)

### Check Shape

In [12]:
ranking_df.shape

(10000, 7)

### Features

In [13]:
features = [
    "similarity_score",
    "popularity_score",
    "recency_score",
    "preference_score"
]

X = ranking_df[features]

y = ranking_df["label"]

### Create Ranker

In [14]:
ranker = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    min_data_in_leaf=5
)

### Train

In [15]:
ranker.fit(
    X,
    y,
    group=groups
)

[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001117 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 561
[LightGBM] [Info] Number of data points in the train set: 10000, number of used features: 4


LGBMRanker(learning_rate=0.05, metric='ndcg', min_data_in_leaf=5,
           objective='lambdarank')

###  Predict

In [16]:
ranking_df["rank_score"] = ranker.predict(X)

[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5


In [17]:
from scipy.special import expit

ranking_df["probability_score"] = expit(
    ranking_df["rank_score"]
)

### Check Scores

In [18]:
ranking_df[
    ["rank_score", "probability_score"]
].head()

,rank_score,probability_score
0,-0.423836,0.395599
1,-0.578865,0.359194
2,-0.020688,0.494828
3,-0.543083,0.367471
4,-0.488459,0.380257


### Top Recommendations

In [19]:
top_recommendations = ranking_df.sort_values(
    "rank_score",
    ascending=False
)

top_recommendations.head(10)

,user_id,item_id,similarity_score,popularity_score,recency_score,preference_score,label,rank_score,probability_score
3044,383,768,0.913673,8,25,0.999711,1,1.249428,0.777201
7249,21,768,0.883957,8,13,0.998740,1,1.246384,0.776673
8688,318,468,0.875692,2,3,0.624594,1,1.233394,0.774412
5152,317,276,0.874488,20,29,0.882127,1,1.212895,0.770811
1004,310,658,0.921205,3,13,0.867344,1,1.195624,0.767745
8619,345,633,0.879808,10,13,0.999963,1,1.186485,0.766112
4540,40,132,0.913839,7,22,0.876762,1,1.134260,0.756624
4142,7,768,0.922943,8,11,0.516132,1,1.087603,0.747930
550,368,786,0.914067,4,10,0.515069,1,1.080158,0.746524
1240,247,659,0.923779,13,22,0.983751,1,1.075504,0.745642


In [20]:
ranking_df["rank_score"] = ranker.predict(X)

[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5


In [21]:
ranking_df.to_csv(
    "data/ranking_results.csv",
    index=False
)